# Day 2 Homework - Ollama Webpage Summarizer

Upgrade the Day 1 website summarizer to use a local open-source model via Ollama.


In [1]:
# Imports
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI
import requests


In [2]:
# Load env vars if you want to reuse any config
load_dotenv(override=True)


True

## Connect to Ollama

Make sure Ollama is running locally before executing the cells below.


In [3]:
# Quick health check (expects Ollama at localhost:11434)
requests.get("http://localhost:11434").status_code


200

In [4]:
# Pull a model if you don't have one yet
!ollama pull llama3.2


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [5]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")


## Prompts and helpers


In [6]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""


In [7]:
def messages_for(website_text):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website_text},
    ]


In [8]:
def summarize(url, model="llama3.2"):
    website_text = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model=model,
        messages=messages_for(website_text),
    )
    return response.choices[0].message.content


In [9]:
def display_summary(url, model="llama3.2"):
    summary = summarize(url, model=model)
    display(Markdown(summary))


In [10]:
display_summary("https://edwarddonner.com")


# Edward Donner's Website: Over Coffee with a Mad Scientist

Edward Donner is a co-founder of Nebula.io, applying AI to help people discover their potential. On the side (aka always), he's a self-taught LLM expert spewing knowledge online. His Udemy courses on AI and machine learning have surprisingly become best-sellers.

News/Announcements:
* January 4, 2026: "AI Builder with n8n – Create Agents and Voice Agents" (new software release)
* November 11, 2025: "The Unique Energy of an AI Live Event" 
* September 15, 2025: "AI Engineering MLOps Track – Deploy AI to Production"

 Warning: too much enthusiasm about LLMs ahead!

In [11]:
display_summary("https://Yallakora.com")

YalaKora seems to be a sports website in the Middle East, focusing on regional and international news, scores, fixtures, and analysis. They cater to fans of various national teams, including Egypt's Al-Ahly team.

Highlights:

* Al-Ahly team has struggled recently in both African and European competitions.
* Egyptian player Mohammad Salah talked about his future at Liverpool, possibly ending his club career.
* A match between English Premier League side Slatan Slough is scheduled to kick off mid-January.